# Kontrola kvality výstupu pipeline (QC) — historie stavebních objektů RÚIAN

Tento notebook analyzuje **výstupní CSV soubory pipeline** (`navrh/output/*_history.csv`)
a kontroluje, jestli pipeline běžela korektně. 

1. základní přehled (rozměry, sloupce, časový rozsah)
2. **chybějící hodnoty** — podíl NaN po sloupcích
3. **konzistence časových sloupců** (`valid_from < valid_to`, `snapshot_date == valid_from`, ...)
4. **návaznost verzí** téhož objektu (mezery/překryvy mezi verzemi, víc než jedna otevřená verze)
5. **rozumnost hodnot** (záporné/extrémní počty podlaží, bytů, zastavěné plochy; kódy mimo číselník)
6. **geometrie** (podíl polygon/jen bod)
7. **geometrie z GeoJSON** (validita, rozsah souřadnic, konzistence s CSV)
8. souhrn podle obce (u souborů za víc obcí)


In [42]:
# --- import knihoven ---

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# --- konfigurace ---

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "analyza" else Path.cwd()

# soubor k analýze:

INPUT_CSV = REPO_ROOT / "navrh" / "output" / "brno_area_history.csv"

# adresář s číselníky RÚIAN 
CODELIST_DIR = REPO_ROOT / "analyza" / "ciselniky"
CODELISTS = {
    "building_type_code": "CS_TYP_STAVEBNIHO_OBJEKTU.csv",
    "usage_type_code": "CE_ZPUSOB_VYUZITI_OBJEKTU.csv",
    "construction_type_code": "CE_DRUH_KONSTRUKCE.csv",
    "sewage_connection_code": "CE_PRIPOJENI_KANAL.csv",
    "gas_connection_code": "CE_PRIPOJENI_PLYNU.csv",
    "water_connection_code": "CE_PRIPOJENI_VODY.csv",
    "elevator_code": "CE_VYBAVENI_VYTAHEM.csv",
    "heating_type_code": "CE_ZPUSOB_VYTAPENI.csv",
}

DATE_COLS = ["record_valid_from", "completion_date", "snapshot_date", "valid_from", "valid_to"]
NUMERIC_COLS = ["unit_count", "floor_count", "built_up_area"]

print(f"Vstupní soubor: {INPUT_CSV}")
assert INPUT_CSV.exists(), f"Soubor neexistuje: {INPUT_CSV}"


Vstupní soubor: c:\Users\kajap\skola\bakalarka\navrh\output\539309_history.csv


In [44]:
# --- načtení ---

def load_history_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


df = load_history_csv(INPUT_CSV)
IS_MULTI_OBEC = "municipality_code" in df.columns
print(f"Načteno {len(df):,} řádků, {df.shape[1]} sloupců.".replace(",", " "))
print(f"Víc obcí v souboru: {IS_MULTI_OBEC}")
df.head(3)


Načteno 3 190 řádků  28 sloupců.
Víc obcí v souboru: False


,gml_id,code,building_type_code,usage_type_code,record_valid_from,completion_date,construction_type_code,unit_count,floor_count,sewage_connection_code,gas_connection_code,water_connection_code,elevator_code,built_up_area,heating_type_code,change_proposal_global_id,transaction_id,iskn_building_id,house_numbers,parcel_ids,district_code,reference_point,has_polygon,snapshot_date,source_file,valid_from,valid_to,end_reason
0,SO.100184871,100184871,3,16.0,2020-02-05,2014-01-20,1.0,0.0,1.0,2.0,3.0,1.0,2.0,757.0,4.0,2199291,3241894,3.306295e+10,NaN,7.209765e+10,NaN,POINT (-756243.38 -1043707.98),True,2020-03-01 00:05:20,20200229_OB_539309_UKSH.xml.zip,2020-03-01 00:05:20,NaT,NaN
1,SO.100687661,100687661,1,7.0,2020-03-26,2020-03-10,1.0,1.0,3.0,1.0,1.0,1.0,2.0,151.0,1.0,2231474,3308065,NaN,953.0,7.750725e+10,55468.0,POINT (-756949.15 -1044497.87),False,2020-04-01 00:13:40,20200331_OB_539309_UKSH.xml.zip,2020-04-01 00:13:40,2020-05-01 00:08:10,change
2,SO.100687661,100687661,1,7.0,2020-03-26,2020-03-10,1.0,1.0,3.0,1.0,1.0,1.0,2.0,151.0,1.0,2231474,3320121,3.635305e+10,953.0,7.750725e+10,55468.0,POINT (-756949.15 -1044497.87),True,2020-05-01 00:08:10,20200430_OB_539309_UKSH.xml.zip,2020-05-01 00:08:10,2022-12-01 00:05:50,change


## 1. Základní přehled

In [45]:
print(f"Unikátních objektů (code): {df['code'].nunique():,}".replace(",", " "))
print(f"Řádků historie celkem:     {len(df):,}".replace(",", " "))
print(f"Verzí na objekt (průměr):  {len(df) / df['code'].nunique():.2f}")
print()
print("Rozsah snapshot_date:", df['snapshot_date'].min(), "→", df['snapshot_date'].max())
print("Počet distinct snapshotů:", df['snapshot_date'].nunique())
print()
print("end_reason:")
print(df['end_reason'].value_counts(dropna=False))
if IS_MULTI_OBEC:
    print()
    print(f"Počet obcí v souboru: {df['municipality_code'].nunique()}")

df.dtypes

Unikátních objektů (code): 1 380
Řádků historie celkem:     3 190
Verzí na objekt (průměr):  2.31

Rozsah snapshot_date: 2015-09-01 00:22:16 → 2026-06-01 00:00:27
Počet distinct snapshotů: 129

end_reason:
end_reason
change     1810
NaN        1340
removed      40
Name: count, dtype: int64


gml_id                                  str
code                                  int64
building_type_code                    int64
usage_type_code                     float64
record_valid_from            datetime64[us]
completion_date              datetime64[us]
construction_type_code              float64
unit_count                          float64
floor_count                         float64
sewage_connection_code              float64
gas_connection_code                 float64
water_connection_code               float64
elevator_code                       float64
built_up_area                       float64
heating_type_code                   float64
change_proposal_global_id             int64
transaction_id                        int64
iskn_building_id                    float64
house_numbers                       float64
parcel_ids                          float64
district_code                       float64
reference_point                         str
has_polygon                     

## 2. Chybějící hodnoty

Dvě verze: po řádcích (verzích) a po objektech (`code`) — u druhé se sloupec
počítá jako chybějící jen když chybí úplně ve všech verzích objektu

### 2a. Na úrovni řádků (verzí)

In [46]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing_tbl = missing.to_frame("% chybí").round(2)
missing_tbl["počet chybí"] = (missing / 100 * len(df)).round().astype(int)
missing_tbl

,% chybí,počet chybí
valid_to,42.01,1340
end_reason,42.01,1340
completion_date,27.27,870
built_up_area,20.09,641
iskn_building_id,12.54,400
house_numbers,9.94,317
district_code,9.94,317
floor_count,8.87,283
elevator_code,8.81,281
construction_type_code,8.34,266


### 2b. Na úrovni objektů (`code`)

In [47]:
notna_any = df.notna().groupby(df["code"]).any()
missing_obj = (100 * (~notna_any).mean()).sort_values(ascending=False)
missing_obj_tbl = missing_obj.to_frame("% chybí").round(2)
missing_obj_tbl["počet chybí"] = (missing_obj / 100 * df["code"].nunique()).round().astype(int)
missing_obj_tbl

,% chybí,počet chybí
valid_to,18.91,261
end_reason,18.91,261
district_code,15.87,219
house_numbers,15.87,219
completion_date,10.36,143
construction_type_code,10.22,141
floor_count,10.07,139
sewage_connection_code,9.49,131
gas_connection_code,9.35,129
water_connection_code,9.35,129


## 3. Konzistence časových sloupců


- `valid_from` nikdy nechybí
- `valid_to IS NULL` právě když `end_reason IS NULL` (objekt je aktuálně otevřený)
- pokud je `valid_to` vyplněné, musí platit `valid_from < valid_to`
- `snapshot_date` verze se musí rovnat jejímu `valid_from` (verze se otevírá přesně
  v okamžiku snapshotu, který ji poprvé zachytil)
- `record_valid_from` (platnost podle katastru) by neměl být pozdější než `valid_from`
  (snapshot, ve kterém jsme verzi poprvé zachytili) 
- `completion_date` (datum dokončení stavby) by mělo být rozumné datum (ne v budoucnosti)

In [48]:
n = len(df)

def report(name, mask, extra_cols=None):
    cnt = int(mask.sum())
    pct = 100 * cnt / n if n else 0
    status = "OK" if cnt == 0 else "POZOR"
    print(f"[{status:5s}] {name}: {cnt:,} řádků ({pct:.3f} %)".replace(",", " "))
    return cnt


report(
    "valid_from chybí", df["valid_from"].isna())

report(
    "valid_to je NULL, ale end_reason ne (nebo naopak)",
    df["valid_to"].isna() != df["end_reason"].isna())

report(
    "valid_from >= valid_to (u uzavřených verzí)",
    df["valid_to"].notna() & (df["valid_from"] >= df["valid_to"]))

report(
    "snapshot_date != valid_from",
    df["snapshot_date"] != df["valid_from"])

report(
    "record_valid_from je pozdější než valid_from (verze 'z budoucnosti')",
    df["record_valid_from"] > df["valid_from"])

current_year = pd.Timestamp.now().year
bad_completion = df["completion_date"].notna() & (
    (df["completion_date"].dt.year < 1000) | (df["completion_date"].dt.year > current_year)
)
report(
    f"completion_date mimo rozsah [1000, {current_year}]", bad_completion)

end_reason_ok = {"change", "removed"}
bad_end_reason = df["end_reason"].notna() & ~df["end_reason"].isin(end_reason_ok)
report(
    f"end_reason mimo {sorted(end_reason_ok)}", bad_end_reason)

[OK   ] valid_from chybí: 0 řádků (0.000 %)
[OK   ] valid_to je NULL  ale end_reason ne (nebo naopak): 0 řádků (0.000 %)
[OK   ] valid_from >= valid_to (u uzavřených verzí): 0 řádků (0.000 %)
[OK   ] snapshot_date != valid_from: 0 řádků (0.000 %)
[OK   ] record_valid_from je pozdější než valid_from (verze 'z budoucnosti'): 0 řádků (0.000 %)
[OK   ] completion_date mimo rozsah [1000  2026]: 0 řádků (0.000 %)
[OK   ] end_reason mimo ['change'  'removed']: 0 řádků (0.000 %)


0

## 4. Návaznost verzí (mezery a překryvy mezi verzemi objektu)

Verze téhož `code` by na sebe měly plynule navazovat —
`valid_to` jedné verze by se mělo rovnat `valid_from` následující verze.
Kontrolujeme:

- **mezery** (`valid_to[i] < valid_from[i+1]`) — něco mezi verzemi chybí
  (typicky výpadek/chybějící měsíční snapshot)
- **překryvy** (`valid_to[i] > valid_from[i+1]`) — verze se časově kryjí
- **duplicitní verze** — stejný `code` a stejný `valid_from` ve dvou řádcích
- **víc než jedna otevřená verze** téhož `code` (víc řádků s `valid_to IS NULL`)

In [49]:
g = df.sort_values(["code", "valid_from"]).groupby("code", sort=False)

# posun o jeden řádek v rámci skupiny -> "další verze"
df_sorted = df.sort_values(["code", "valid_from"]).reset_index(drop=True)
next_valid_from = df_sorted.groupby("code")["valid_from"].shift(-1)
same_code_next = df_sorted["code"] == df_sorted["code"].shift(-1)
this_valid_to = df_sorted["valid_to"]

has_gap = same_code_next & this_valid_to.notna() & next_valid_from.notna() & (this_valid_to < next_valid_from)
has_overlap = same_code_next & this_valid_to.notna() & next_valid_from.notna() & (this_valid_to > next_valid_from)

gap_df = df_sorted.loc[has_gap, ["code", "valid_to"]].copy()
gap_df["next_valid_from"] = next_valid_from[has_gap]
gap_df["gap_days"] = (gap_df["next_valid_from"] - gap_df["valid_to"]).dt.days

overlap_df = df_sorted.loc[has_overlap, ["code", "valid_to"]].copy()
overlap_df["next_valid_from"] = next_valid_from[has_overlap]
overlap_df["overlap_days"] = (overlap_df["valid_to"] - overlap_df["next_valid_from"]).dt.days

print(f"Mezery mezi verzemi:  {len(gap_df):,}")
print(f"Překryvy mezi verzemi: {len(overlap_df):,}")


if len(gap_df):
    print("\nRozdělení délky mezer (dny):")
    print(gap_df["gap_days"].describe().round(1))
    print("\nNejdelší mezery:")
    display(gap_df.sort_values("gap_days", ascending=False).head(10))

Mezery mezi verzemi:  0
Překryvy mezi verzemi: 0


In [50]:
# duplicitní verze (stejný code + stejný valid_from)
dup_mask = df.duplicated(subset=["code", "valid_from"], keep=False)
dup_df = df.loc[dup_mask, ["code", "valid_from", "valid_to", "end_reason"]].sort_values(["code", "valid_from"])
report("duplicitní dvojice (code, valid_from)", dup_mask)
if dup_mask.any():
    display(dup_df.head(20))

[OK   ] duplicitní dvojice (code  valid_from): 0 řádků (0.000 %)


In [51]:
# víc než jedna otevřená verze na objekt
open_counts = df[df["valid_to"].isna()].groupby("code").size()
multi_open = open_counts[open_counts > 1]
print(f"Objektů s víc než jednou otevřenou verzí: {len(multi_open):,}".replace(",", " "))
if len(multi_open):
    display(multi_open.sort_values(ascending=False).head(10).to_frame("počet otevřených verzí"))

Objektů s víc než jednou otevřenou verzí: 0


In [52]:
# kolik objektů již zaniklo a kontrola, zda některé zanikly víckrát 
episodes = df[df["end_reason"] == "removed"].groupby("code").size()
print(f"Objektů, které aspoň jednou zanikly (end_reason='removed'): {episodes.shape[0]:,}".replace(",", " "))
print(f"Z toho s víc než jedním zánikem (>1x removed): {(episodes > 1).sum():,}".replace(",", " "))

Objektů  které aspoň jednou zanikly (end_reason='removed'): 40
Z toho s víc než jedním zánikem (>1x removed): 0


## 5. Rozmezí hodnot 

### 5a. Číselné atributy

`unit_count` (počet bytů), `floor_count` (počet podlaží) a `built_up_area`
(zastavěná plocha v m²) by měly být nezáporné a v rozumném rozsahu

In [53]:
desc = df[NUMERIC_COLS].describe(percentiles=[.01, .5, .9, .99, .999]).T
desc

,count,mean,std,min,1%,50%,90%,99%,99.9%,max
unit_count,3005.0,1.749085,5.047867,0.0,0.00,1.0,2.0,30.0,40.000,110.0
floor_count,2907.0,1.842449,0.714090,1.0,1.00,2.0,2.0,4.0,5.000,6.0
built_up_area,2549.0,156.150647,316.016663,3.0,13.92,114.0,208.0,1035.0,4356.572,9163.0


In [54]:
SANITY_RULES = {
    # min povolené, max "rozumné" hodnoty - jen orientační horní mez)
    "unit_count": (0, 1000),
    "floor_count": (0, 40),       
    "built_up_area": (0, 100_000),  
}

for col, (lo, hi) in SANITY_RULES.items():
    below = df[col].notna() & (df[col] < lo)
    above = df[col].notna() & (df[col] > hi)
    report(f"{col} < {lo}", below)
    report(f"{col} > {hi} (orientační horní mez)", above)
    if above.any():
        display(df.loc[above, ["code", col, "snapshot_date"]].sort_values(col, ascending=False).head(5))

[OK   ] unit_count < 0: 0 řádků (0.000 %)
[OK   ] unit_count > 1000 (orientační horní mez): 0 řádků (0.000 %)
[OK   ] floor_count < 0: 0 řádků (0.000 %)
[OK   ] floor_count > 40 (orientační horní mez): 0 řádků (0.000 %)
[OK   ] built_up_area < 0: 0 řádků (0.000 %)
[OK   ] built_up_area > 100000 (orientační horní mez): 0 řádků (0.000 %)


### 5b. Kódované atributy proti číselníkům RÚIAN

Všech osm kódovaných atributů má oficiální
číselník RÚIAN stažený v `analyza/ciselniky/` (staženo ze
`services.cuzk.gov.cz/sestavy/cis/`) 


In [55]:
def load_codelist(fname: str) -> set:
    path = CODELIST_DIR / fname
    if not path.exists():
        print(f"  (číselník {fname} nenalezen, sekce se přeskakuje)")
        return None
    cl = pd.read_csv(path, sep=";", encoding="cp1250")
    return set(cl["KOD"].astype(int))


for col, fname in CODELISTS.items():
    if col not in df.columns:
        continue
    valid_codes = load_codelist(fname)
    if valid_codes is None:
        continue
    present = df[col].dropna().astype(int)
    bad = present[~present.isin(valid_codes)]
    pct = 100 * len(bad) / len(present) if len(present) else 0
    status = "OK" if len(bad) == 0 else "POZOR"
    print(f"[{status:5s}] {col}: {len(bad):,} hodnot mimo číselník ({pct:.3f} %)".replace(",", " "))
    if len(bad):
        display(bad.value_counts().head(10).to_frame("počet"))

[OK   ] building_type_code: 0 hodnot mimo číselník (0.000 %)
[OK   ] usage_type_code: 0 hodnot mimo číselník (0.000 %)
[OK   ] construction_type_code: 0 hodnot mimo číselník (0.000 %)
[OK   ] sewage_connection_code: 0 hodnot mimo číselník (0.000 %)
[OK   ] gas_connection_code: 0 hodnot mimo číselník (0.000 %)
[OK   ] water_connection_code: 0 hodnot mimo číselník (0.000 %)
[OK   ] elevator_code: 0 hodnot mimo číselník (0.000 %)
[OK   ] heating_type_code: 0 hodnot mimo číselník (0.000 %)


## 6. Geometrie — polygon vs. jen definiční bod

`has_polygon` říká, jestli měl objekt v daném snapshotu celou geometrii půdorysu
(`OriginalniHranice`), nebo jen definiční bod.

In [56]:
print(f"Celkový podíl řádků s polygonem: {df['has_polygon'].mean() * 100:.1f} %")
print(f"Podíl řádků s chybějícím reference_point: {df['reference_point'].isna().mean() * 100:.2f} %")

# per objekt (code): má aspoň jedna z jeho verzí polygon? 
obj_has_polygon = df.groupby("code")["has_polygon"].any()
print(f"Podíl objektů s polygonem aspoň v jedné verzi: {obj_has_polygon.mean() * 100:.1f} % "
      f"({obj_has_polygon.sum():,} z {len(obj_has_polygon):,})".replace(",", " "))

Celkový podíl řádků s polygonem: 87.1 %
Podíl řádků s chybějícím reference_point: 0.75 %
Podíl objektů s polygonem aspoň v jedné verzi: 97.0 % (1 339 z 1 380)


## 7. Geometrie z GeoJSON — validita a rozsah souřadnic

CSV výstup neobsahuje geometrii samotnou (jen odvozené atributy `has_polygon`/`reference_point`) 

Geometrii obsahuje odpovídající `*_history.geojson` 

- **chybějící/prázdná geometrie** 
- **validita geometrie** (`is_valid`) — self-intersections a jiné
  topologické chyby polygonů
- **rozsah souřadnic** — platný rozsah WGS84 a orientačně bounding
  box ČR (geojson je exportovaný do EPSG:4326, nativní souřadnice pipeliny jsou v EPSG:5514 — S-JTSK / Krovak East North)
- **degenerované polygony** (nulová plocha)
- **konzistence s CSV** — `has_polygon` musí odpovídat skutečnému typu
  geometrie (Polygon/MultiPolygon vs. Point)

In [57]:
GEOJSON_PATH = INPUT_CSV.parent / (INPUT_CSV.stem + ".geojson")
print(f"GeoJSON soubor: {GEOJSON_PATH}")

if not GEOJSON_PATH.exists():
    print("  (geojson nenalezen, sekce 9 se přeskakuje)")
    geo = None
else:
    geo = gpd.read_file(GEOJSON_PATH)
    geo["valid_from"] = pd.to_datetime(geo["valid_from"], errors="coerce")
    geo["code"] = geo["code"].astype(df["code"].dtype)  

    n_geo = len(geo)
    print(f"Načteno {n_geo:,} řádků geometrie, CRS: {geo.crs}".replace(",", " "))
    if n_geo != len(df):
        print(f"[POZOR] počet řádků v geojson ({n_geo}) != počet řádků v CSV ({len(df)}) "
              f"- napojení podle (code, valid_from) níž nemusí sedět 1:1")

    def report_geo(name, mask):
        cnt = int(mask.sum())
        pct = 100 * cnt / n_geo if n_geo else 0
        status = "OK" if cnt == 0 else "POZOR"
        print(f"[{status:5s}] {name}: {cnt:,} řádků ({pct:.3f} %)".replace(",", " "))
        return cnt

    report_geo("chybějící/prázdná geometrie (ani definiční bod)",
               geo.geometry.isna() | geo.geometry.is_empty)

GeoJSON soubor: c:\Users\kajap\skola\bakalarka\navrh\output\539309_history.geojson
Načteno 3 190 řádků geometrie  CRS: EPSG:4326
[POZOR] chybějící/prázdná geometrie (ani definiční bod): 17 řádků (0.533 %)


In [58]:
if geo is not None:
    valid_geo = geo[geo.geometry.notna() & ~geo.geometry.is_empty].copy()

    report_geo("neplatná geometrie (self-intersection apod., is_valid == False)",
               ~valid_geo.geometry.is_valid)

    invalid = valid_geo[~valid_geo.geometry.is_valid]
    if len(invalid):
        from shapely.validation import explain_validity
        display(invalid.assign(duvod=invalid.geometry.map(explain_validity))
                 [["code", "valid_from", "duvod"]].head(10))

    assert str(geo.crs) == "EPSG:4326", \
        f"Neočekávané CRS geojsonu: {geo.crs} (export_tabular.py by měl exportovat do EPSG:4326)"

    bounds = valid_geo.geometry.bounds  

    bad_wgs84 = (bounds[["minx", "maxx"]].abs().max(axis=1) > 180) | (bounds[["miny", "maxy"]].abs().max(axis=1) > 90)
    report_geo("souřadnice mimo platný rozsah WGS84 ([-180,180] x [-90,90])", bad_wgs84)

    # orientační bounding box ČR 
    CZ_LON = (11.8, 19.1)
    CZ_LAT = (48.4, 51.2)
    bad_cz = (
        (bounds["minx"] < CZ_LON[0]) | (bounds["maxx"] > CZ_LON[1]) |
        (bounds["miny"] < CZ_LAT[0]) | (bounds["maxy"] > CZ_LAT[1])
    )
    report_geo(f"souřadnice mimo orientační rozsah ČR (lon {CZ_LON}, lat {CZ_LAT})", bad_cz)
    if bad_cz.any():
        display(valid_geo.loc[bad_cz, ["code", "valid_from"]].join(bounds[bad_cz]).head(10))

    # nulová plocha u polygonů 
    polys = valid_geo[valid_geo.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    polys_proj = polys.to_crs("EPSG:5514")
    report_geo("polygon s nulovou plochou (degenerovaná geometrie)", polys_proj.geometry.area == 0)

[OK   ] neplatná geometrie (self-intersection apod.  is_valid == False): 0 řádků (0.000 %)
[OK   ] souřadnice mimo platný rozsah WGS84 ([-180 180] x [-90 90]): 0 řádků (0.000 %)
[OK   ] souřadnice mimo orientační rozsah ČR (lon (11.8  19.1)  lat (48.4  51.2)): 0 řádků (0.000 %)
[OK   ] polygon s nulovou plochou (degenerovaná geometrie): 0 řádků (0.000 %)


In [59]:
if geo is not None:
    # konzistence mezi CSV atributem has_polygon a skutečným typem geometrie v geojsonu
    merged = df.merge(
        geo[["code", "valid_from", "geometry"]],
        on=["code", "valid_from"], how="left",
    )
    is_polygonish = merged["geometry"].apply(
        lambda g: g is not None and not g.is_empty and g.geom_type in ("Polygon", "MultiPolygon")
    )
    mismatch = merged["has_polygon"] != is_polygonish
    report("has_polygon (CSV) neodpovídá skutečnému typu geometrie v geojsonu", mismatch)
    if mismatch.any():
        display(merged.loc[mismatch, ["code", "valid_from", "has_polygon"]].assign(
            geom_type=merged.loc[mismatch, "geometry"].map(lambda g: g.geom_type if g is not None else None)
        ).head(10))

[OK   ] has_polygon (CSV) neodpovídá skutečnému typu geometrie v geojsonu: 0 řádků (0.000 %)


## 8. Souhrn podle obce

Všechny sloupce jsou **na úrovni objektu**, ne
řádku (jen když daný atribut chybí
ve všech jeho verzích)

- `chybi_dokonceni_pct` — objekt nemá `completion_date` v žádné verzi
- `chybi_polygon_pct` — objekt nemá půdorys (`OriginalniHranice`) v žádné
  verzi
- `chybi_geometrie_pct` — objekt nemá v žádné verzi ani polygon,
  ani definiční bod

In [ ]:
obec_cols = ["municipality_code", "municipality_name"] if IS_MULTI_OBEC else []

OBEC_NAZEV_LOOKUP = {"539309": "Chýně"}

dokonceni_chybi_obj = ~df.notna().groupby(df["code"])["completion_date"].any()

polygon_chybi_obj = ~df.groupby("code")["has_polygon"].any()

if geo is not None:
    geom_missing = merged["geometry"].isna() | merged["geometry"].apply(lambda g: g is not None and g.is_empty)
    geometrie_chybi_obj = geom_missing.groupby(merged["code"]).all()
else:
    geometrie_chybi_obj = None
    print("  (geojson ze sekce 8 nenačten - chybi_geometrie_pct se vynechá)")

obj_tbl = df.drop_duplicates("code").set_index("code")[obec_cols]
obj_tbl = obj_tbl.assign(chybi_dokonceni=dokonceni_chybi_obj,
                         chybi_polygon=polygon_chybi_obj)
if geometrie_chybi_obj is not None:
    obj_tbl["chybi_geometrie"] = geometrie_chybi_obj

group_cols = obec_cols or [lambda _: "celkem"]
gb = obj_tbl.groupby(group_cols)

per_obec = gb.agg(
    objektu=("chybi_dokonceni", "size"),
    chybi_dokonceni_pct=("chybi_dokonceni", lambda s: round(100 * s.mean(), 1)),
    chybi_polygon_pct=("chybi_polygon", lambda s: round(100 * s.mean(), 1)),
)
if geometrie_chybi_obj is not None:
    per_obec["chybi_geometrie_pct"] = gb["chybi_geometrie"].apply(lambda s: round(100 * s.mean(), 1))

radku = df.groupby(group_cols).size().rename("radku") if obec_cols else pd.Series({"celkem": len(df)}, name="radku")
per_obec = radku.to_frame().join(per_obec).reset_index()
if not obec_cols:
    per_obec = per_obec.drop(columns=["index"])
    kody = df["source_file"].str.extract(r"_OB_(\d+)_UKSH")[0].dropna()
    obec_kod = kody.mode().iat[0] if not kody.empty else "?"
    per_obec.insert(0, "municipality_code", obec_kod)
    per_obec.insert(1, "municipality_name", OBEC_NAZEV_LOOKUP.get(obec_kod, "?"))
per_obec = per_obec.sort_values("radku", ascending=False)

print(f"Počet obcí v souboru: {len(per_obec)}")
display(per_obec)

Počet obcí v souboru: 1


,obec_kod,obec_nazev,radku,objektu,chybi_dokonceni_pct,chybi_polygon_pct,chybi_geometrie_pct
0,539309,Chýně,3190,1380,10.4,3.0,1.0
